In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder, OneHotEncoder
import pickle

In [23]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [24]:
# Preprocess the data
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [25]:
# Encode categorical variables
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])

In [26]:
# One-hot encode the 'Geography' column
one_hot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = one_hot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [27]:
# Concatenate the one-hot encoded columns with the original dataframe
data = pd.concat([data, geo_encoded_df], axis=1)
data.drop(['Geography'], axis=1, inplace=True)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [28]:
# Split the data into features and target variable
x = data.drop(['EstimatedSalary'], axis=1)
y = data['EstimatedSalary']

In [29]:
# Split the data train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [30]:
# Scale the features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


StandardScaler kullanırken fit_transform ve transform arasındaki fark, veri kümesinin ölçeklendirme parametrelerini öğrenme sürecinde ortaya çıkar:

**fit_transform:**

fit işlemi, verilen veri kümesinin ortalama ve standart sapma değerlerini hesaplar.
Daha sonra, bu hesaplanan değerlere göre veri kümesini ölçeklendirir.
Bu işlem, yalnızca eğitim veri kümesi (x_train) üzerinde yapılır, çünkü modelin test verilerini görmemesi gerekir.
transform:

transform işlemi, daha önce fit ile öğrenilen ortalama ve standart sapma değerlerini kullanarak verilen veri kümesini ölçeklendirir.
Test veri kümesi (x_test) üzerinde transform kullanılır, çünkü test verisinin ölçeklendirilmesi sırasında eğitim verisinden öğrenilen parametreler kullanılmalıdır.
Neden Bu Şekilde Kullanıyoruz?
Eğitim Verisi (x_train): fit_transform kullanılır, çünkü eğitim verisinin ortalama ve standart sapma değerleri bu aşamada öğrenilir.

Test Verisi (x_test): transform kullanılır, çünkü test verisi, eğitim verisinden öğrenilen parametrelerle ölçeklendirilmelidir. Test verisinin özellikleri (ortalama ve standart sapma) öğrenilmez; aksi takdirde veri sızıntısı (data leakage) olur ve modelin gerçek performansı yanlış değerlendirilir.

**Özet:**
fit_transform: Eğitim verisi için kullanılır (ölçeklendirme parametrelerini öğrenir ve uygular).
transform: Test verisi için kullanılır (önceden öğrenilen parametreleri uygular).


In [31]:
# Save the encoders and scaler for future use
with open('label_encoder_gender.pkl', 'wb') as le_file:
    pickle.dump(label_encoder, le_file)

with open('one_hot_encoder_geo.pkl', 'wb') as ohe_file:
    pickle.dump(one_hot_encoder_geo, ohe_file)

with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)





In [32]:
#### ANN Regression Model ####
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [33]:
# Build the ANN model
model = Sequential()
model.add(Dense(units=64, activation='relu', input_shape=(x_train.shape[1],)))
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=16, activation='relu'))
model.add(Dense(units=1, activation='linear'))  # Output layer for regression

# Compile the model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])
model.summary()


c:\dev\Python\.venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,457 (13.50 KB)

 Trainable params: 3,457 (13.50 KB)

 Non-trainable params: 0 (0.00 B)

Bu kod, bir Yapay Sinir Ağı (Artificial Neural Network - ANN) modelini oluşturur, derler ve özetini görüntüler. İşte adım adım açıklaması:

1. Modelin Tanımlanması
Sequential: Keras'ta katmanları sıralı bir şekilde eklemek için kullanılan bir model türüdür. Bu, her katmanın bir önceki katmanın çıktısını girdi olarak aldığı basit bir modeldir.
2. Katmanların Eklenmesi
Dense: Tam bağlantılı (fully connected) bir katman ekler.
units=64: Bu katmanda 64 nöron bulunur.
activation='relu': Aktivasyon fonksiyonu olarak ReLU (Rectified Linear Unit) kullanılır. Bu, negatif değerleri sıfıra eşitler ve pozitif değerleri olduğu gibi bırakır.
input_shape=(x_train.shape[1],): Girdi verisinin şekli belirtilir. Burada, x_train'in sütun sayısı (özellik sayısı) kullanılır.
Bu iki satır, sırasıyla 32 ve 16 nöronlu iki gizli katman ekler. Her iki katmanda da ReLU aktivasyon fonksiyonu kullanılır.
Çıkış katmanı eklenir.
units=1: Çıkışta yalnızca bir nöron bulunur (tek bir sürekli değer tahmini için, yani regresyon problemi).
activation='linear': Aktivasyon fonksiyonu olarak doğrusal (linear) fonksiyon kullanılır. Bu, regresyon problemleri için uygundur.
3. Modelin Derlenmesi
optimizer='adam': Modelin ağırlıklarını optimize etmek için Adam optimizasyon algoritması kullanılır. Bu, yaygın olarak kullanılan ve genellikle iyi sonuçlar veren bir optimizasyon algoritmasıdır.
loss='mean_absolute_error': Kayıp fonksiyonu olarak Ortalama Mutlak Hata (Mean Absolute Error - MAE) kullanılır. Bu, tahmin edilen değerler ile gerçek değerler arasındaki farkların mutlak değerlerinin ortalamasını hesaplar.
metrics=['mae']: Eğitim sırasında modelin performansını değerlendirmek için MAE metriği kullanılır.
4. Modelin Özeti
Modelin yapısını, katmanlarını, her katmandaki parametre sayısını ve toplam parametre sayısını özetler.


In [34]:
# Setup tensorboard callback
from tensorflow.keras.callbacks import TensorBoard,EarlyStopping
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)



In [35]:
# Setup early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


In [36]:
# Train the model
history = model.fit(x_train,
                    y_train, 
                    epochs=100,
                    validation_data=(x_test, y_test),
                    callbacks=[tensorboard_callback, early_stopping_callback],
                    )


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 99454.8672 - mae: 99454.8672 - val_loss: 96043.7891 - val_mae: 96043.7891
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 92747.6484 - mae: 92747.6484 - val_loss: 66098.4922 - val_mae: 66098.4922
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 60065.4570 - mae: 60065.4570 - val_loss: 50318.9531 - val_mae: 50318.9531
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 49753.8945 - mae: 49753.8945 - val_loss: 50235.4062 - val_mae: 50235.4062
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 49861.1328 - mae: 49861.1328 - val_loss: 50227.0117 - val_mae: 50227.0117
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 49900.0039 - mae: 49900.0039 - val_loss: 50244.0469 - val_mae: 50244.0469
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 49398.3242 - mae: 49398.3242 - val_loss: 50279.9609 - val_mae: 50279.9609
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - 

In [38]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit/ --bind_all --port 6006

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 27692), started 0:00:31 ago. (Use '!kill 27692' to kill it.)

In [39]:
# Evaluate the model on the test set
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f"Test MAE: {test_mae:.2f}")


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 51128.2578 - mae: 51128.2578
Test MAE: 50209.11


In [41]:
# Model save
model.save('ann_regression_model.keras')